# 02 — Demand Forecasting

Coffee Retail MLOps  
Forecast daily revenue per product using three model families:
1. Seasonal Naive (baseline)
2. Prophet (weekly seasonality)
3. XGBoost with lag features (champion candidate)

**Hold-out:** last 30 days of the dataset (temporal split — no random shuffle)  
**Primary metric:** MAPE  
**MLflow experiment:** `demand_forecasting`

---

## Setup — imports, paths, MLflow

In [ ]:
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import mlflow
import mlflow.sklearn
import mlflow.xgboost

# ── project root (notebook lives in coffee_sales/notebooks/) ──
NOTEBOOK_DIR = Path(".").resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent

# add src to path so we can import project modules
sys.path.insert(0, str(PROJECT_ROOT))

from src.features.demand_features import build_features, run as build_and_save_features, FORECAST_PRODUCTS
from src.models.demand.evaluate import evaluate, evaluate_prophet, _mape
from src.models.demand.train import train, predict, ALL_FEATURES

# ── MLflow tracking ──
MLRUNS_DIR = PROJECT_ROOT / "mlruns"
MLRUNS_DIR.mkdir(exist_ok=True)
mlflow.set_tracking_uri(f"file://{MLRUNS_DIR}")
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
mlflow.set_tracking_uri(f"sqlite:///{PROJECT_ROOT / 'mlflow.db'}")
mlflow.set_experiment("demand_forecasting")

# ── output paths ──
FEATURES_PATH = PROJECT_ROOT / "data" / "features" / "demand_features.parquet"
PREDICTIONS_PATH = PROJECT_ROOT / "data" / "predictions" / "demand_forecast.parquet"
PREDICTIONS_PATH.parent.mkdir(parents=True, exist_ok=True)

HOLD_OUT_DAYS = 30
FORECAST_HORIZON_7  = 7
FORECAST_HORIZON_30 = 30
SEED = 42

print("Project root:", PROJECT_ROOT)
print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Forecast products:", FORECAST_PRODUCTS)

---
## Section 1 — Load Feature Store & Visualise Series

In [ ]:
# Build (or re-use) the feature store
if not FEATURES_PATH.exists():
    print("Building demand features from scratch...")
    build_and_save_features(output_path=FEATURES_PATH)

df_feat = pd.read_parquet(FEATURES_PATH, engine="pyarrow")
df_feat["date"] = pd.to_datetime(df_feat["date"])

print(f"Feature store: {len(df_feat):,} rows | {df_feat['product'].nunique()} products")
print(f"Date range: {df_feat['date'].min().date()} → {df_feat['date'].max().date()}")
df_feat.head()

In [ ]:
# Plot daily revenue series for all 8 products
fig, axes = plt.subplots(4, 2, figsize=(16, 14), sharex=True)
axes = axes.flatten()

for ax, product in zip(axes, FORECAST_PRODUCTS):
    sub = df_feat[df_feat["product"] == product].sort_values("date")
    ax.plot(sub["date"], sub["revenue"], linewidth=0.8, color="steelblue")
    ax.set_title(product, fontsize=10)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b'%y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.tick_params(axis="x", rotation=30, labelsize=7)
    ax.tick_params(axis="y", labelsize=7)

fig.suptitle("Daily Revenue by Product", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()
print("Total revenue per product:")
df_feat.groupby("product")["revenue"].sum().sort_values(ascending=False).apply(lambda v: f"R$ {v:,.0f}")

---
## Section 2 — Train / Test Split (Hold-out = last 30 days)

In [ ]:
max_date = df_feat["date"].max()
cutoff_date = max_date - pd.Timedelta(days=HOLD_OUT_DAYS - 1)

df_train_all = df_feat[df_feat["date"] < cutoff_date].copy()
df_test_all  = df_feat[df_feat["date"] >= cutoff_date].copy()

print(f"Cutoff date : {cutoff_date.date()}")
print(f"Train rows  : {len(df_train_all):,}  ({df_train_all['date'].min().date()} → {df_train_all['date'].max().date()})")
print(f"Test rows   : {len(df_test_all):,}  ({df_test_all['date'].min().date()} → {df_test_all['date'].max().date()})")

---
## Section 3 — Baseline: Seasonal Naive

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

def seasonal_naive_predict(df_train: pd.DataFrame, df_test: pd.DataFrame, season: int = 7) -> np.ndarray:
    """
    Seasonal Naive: predict each test day as the revenue of
    the same day of the week from the last available training week.
    """
    predictions = []
    train_sorted = df_train.sort_values("date")
    for _, row in df_test.iterrows():
        target_dow = row["date"].dayofweek
        # find the most recent training day with same day-of-week
        same_dow = train_sorted[train_sorted["date"].dt.dayofweek == target_dow]
        if len(same_dow) == 0:
            predictions.append(train_sorted["revenue"].mean())
        else:
            predictions.append(same_dow.iloc[-1]["revenue"])
    return np.array(predictions)


naive_results = {}

with mlflow.start_run(run_name="seasonal_naive_baseline"):
    mlflow.set_tag("model_type", "seasonal_naive")
    all_mape, all_mae, all_rmse = [], [], []

    for product in FORECAST_PRODUCTS:
        df_tr = df_train_all[df_train_all["product"] == product].copy()
        df_te = df_test_all[df_test_all["product"] == product].copy()

        y_pred = seasonal_naive_predict(df_tr, df_te)
        y_true = df_te["revenue"].values

        mae  = float(mean_absolute_error(y_true, y_pred))
        rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
        mape = _mape(y_true, y_pred)

        naive_results[product] = {"mae": mae, "rmse": rmse, "mape": mape}
        all_mape.append(mape); all_mae.append(mae); all_rmse.append(rmse)

        mlflow.log_metrics({
            f"{product.replace(' ','_')}_mae":  mae,
            f"{product.replace(' ','_')}_rmse": rmse,
            f"{product.replace(' ','_')}_mape": mape,
        })
        print(f"  {product:<25} MAE={mae:6.1f}  RMSE={rmse:6.1f}  MAPE={mape:5.1f}%")

    avg_mape = float(np.nanmean(all_mape))
    avg_mae  = float(np.nanmean(all_mae))
    avg_rmse = float(np.nanmean(all_rmse))
    mlflow.log_metrics({"avg_mape": avg_mape, "avg_mae": avg_mae, "avg_rmse": avg_rmse})
    print(f"\nSeasonal Naive — avg MAPE={avg_mape:.1f}%  avg MAE={avg_mae:.1f}  avg RMSE={avg_rmse:.1f}")

---
## Section 4 — Prophet (weekly seasonality)

In [ ]:
PROPHET_AVAILABLE = False
try:
    from prophet import Prophet
    PROPHET_AVAILABLE = True
    print("Prophet is available — running Prophet models")
except ImportError:
    print("⚠  Prophet not installed — skipping Section 4. XGBoost will serve as champion.")

prophet_results = {}

In [ ]:
if PROPHET_AVAILABLE:
    import logging as _logging
    _logging.getLogger("prophet").setLevel(_logging.WARNING)
    _logging.getLogger("cmdstanpy").setLevel(_logging.WARNING)

    for product in FORECAST_PRODUCTS:
        df_tr = df_train_all[df_train_all["product"] == product][["date", "revenue"]].copy()
        df_te = df_test_all[df_test_all["product"] == product][["date", "revenue"]].copy()

        df_tr_prophet = df_tr.rename(columns={"date": "ds", "revenue": "y"})

        with mlflow.start_run(run_name=f"prophet_{product.replace(' ','_')}", nested=False):
            mlflow.set_tag("model_type", "prophet")
            mlflow.set_tag("product", product)

            model = Prophet(
                weekly_seasonality=True,
                yearly_seasonality=True,
                daily_seasonality=False,
                changepoint_prior_scale=0.05,
                seasonality_prior_scale=10,
                interval_width=0.95,
            )
            model.fit(df_tr_prophet)

            future = pd.DataFrame({"ds": df_te["date"].values})
            forecast = model.predict(future)
            y_pred = np.clip(forecast["yhat"].values, 0, None)
            y_true = df_te["revenue"].values

            mae  = float(mean_absolute_error(y_true, y_pred))
            rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
            mape = _mape(y_true, y_pred)

            prophet_results[product] = {"mae": mae, "rmse": rmse, "mape": mape}
            mlflow.log_metrics({"mae": mae, "rmse": rmse, "mape": mape})
            mlflow.log_params({"weekly_seasonality": True, "yearly_seasonality": True,
                               "changepoint_prior_scale": 0.05})

            print(f"  {product:<25} MAE={mae:6.1f}  RMSE={rmse:6.1f}  MAPE={mape:5.1f}%")

    avg_mape_p = float(np.nanmean([v["mape"] for v in prophet_results.values()]))
    print(f"\nProphet — avg MAPE={avg_mape_p:.1f}%")
else:
    print("Prophet skipped.")

---
## Section 5 — XGBoost with Lag Features

In [ ]:
import xgboost as xgb

xgb_results = {}
xgb_models  = {}

DEFAULT_XGB_PARAMS = {
    "n_estimators":    200,
    "max_depth":       4,
    "learning_rate":   0.05,
    "subsample":       0.8,
    "colsample_bytree":0.8,
    "random_state":    SEED,
    "n_jobs":          -1,
    "verbosity":       0,
}

for product in FORECAST_PRODUCTS:
    df_tr = df_train_all[df_train_all["product"] == product].copy()
    df_te = df_test_all[df_test_all["product"] == product].copy()

    run_name = f"xgboost_lag_{product.replace(' ','_')}"
    with mlflow.start_run(run_name=run_name):
        mlflow.set_tag("model_type", "xgboost_lag")
        mlflow.set_tag("product", product)
        mlflow.log_params(DEFAULT_XGB_PARAMS)

        model, train_metrics = train(product, df_tr, DEFAULT_XGB_PARAMS.copy())
        test_metrics = evaluate(model, df_te, product)

        mlflow.log_metrics({
            "mae":  test_metrics["mae"],
            "rmse": test_metrics["rmse"],
            "mape": test_metrics["mape"],
            "train_mae":  train_metrics["mae"],
            "train_rmse": train_metrics["rmse"],
        })
        mlflow.xgboost.log_model(model, artifact_path="model")

        xgb_results[product] = test_metrics
        xgb_models[product]  = model

    print(f"  {product:<25} MAE={test_metrics['mae']:6.1f}  RMSE={test_metrics['rmse']:6.1f}  MAPE={test_metrics['mape']:5.1f}%")

avg_mape_xgb = float(np.nanmean([v["mape"] for v in xgb_results.values()]))
print(f"\nXGBoost lag — avg MAPE={avg_mape_xgb:.1f}%")

---
## Section 6 — Champion Selection (lowest avg MAPE)

In [ ]:
comparison_rows = []
for product in FORECAST_PRODUCTS:
    row = {"product": product}
    row["naive_mape"]  = naive_results.get(product, {}).get("mape", float("nan"))
    if PROPHET_AVAILABLE:
        row["prophet_mape"] = prophet_results.get(product, {}).get("mape", float("nan"))
    row["xgb_mape"]    = xgb_results.get(product, {}).get("mape", float("nan"))
    comparison_rows.append(row)

df_compare = pd.DataFrame(comparison_rows).set_index("product")

# avg MAPE per model
avg_row = df_compare.mean(numeric_only=True).rename("AVERAGE")
df_compare_display = pd.concat([df_compare, avg_row.to_frame().T])
print("\n=== Model Comparison — MAPE (%) ===\n")
print(df_compare_display.round(1).to_string())

# Champion = column with lowest average MAPE
mape_avgs = {}
mape_avgs["Seasonal Naive"] = float(np.nanmean([naive_results.get(p, {}).get("mape", float("nan")) for p in FORECAST_PRODUCTS]))
if PROPHET_AVAILABLE and prophet_results:
    mape_avgs["Prophet"] = float(np.nanmean([prophet_results.get(p, {}).get("mape", float("nan")) for p in FORECAST_PRODUCTS]))
mape_avgs["XGBoost Lag"] = avg_mape_xgb

CHAMPION_NAME = min(mape_avgs, key=mape_avgs.get)
print(f"\n📌 Champion model: {CHAMPION_NAME} (avg MAPE={mape_avgs[CHAMPION_NAME]:.1f}%)")
for name, val in sorted(mape_avgs.items(), key=lambda x: x[1]):
    marker = " ← CHAMPION" if name == CHAMPION_NAME else ""
    print(f"  {name:<20} avg MAPE={val:.1f}%{marker}")

---
## Section 7 — Optuna Tuning of the Champion (≥ 20 trials)

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

N_TRIALS = 20

print(f"Tuning champion ({CHAMPION_NAME}) with Optuna — {N_TRIALS} trials per product")
print("This may take a few minutes...\n")

best_params_per_product: dict = {}
tuned_models: dict = {}
tuned_results: dict = {}

def make_objective(df_tr: pd.DataFrame, df_te: pd.DataFrame):
    """Return an Optuna objective closure for one product."""
    def objective(trial: optuna.Trial) -> float:
        params = {
            "n_estimators":     trial.suggest_int("n_estimators", 100, 600),
            "max_depth":        trial.suggest_int("max_depth", 2, 8),
            "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "subsample":        trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "gamma":            trial.suggest_float("gamma", 0.0, 5.0),
            "random_state":     SEED,
            "n_jobs":           -1,
            "verbosity":        0,
        }
        import xgboost as _xgb
        from sklearn.metrics import mean_absolute_error as _mae
        X_tr = df_tr[ALL_FEATURES]; y_tr = df_tr["revenue"].values
        X_te = df_te[ALL_FEATURES]; y_te = df_te["revenue"].values
        m = _xgb.XGBRegressor(**params)
        m.fit(X_tr, y_tr)
        y_pred = np.clip(m.predict(X_te), 0, None)
        from src.models.demand.evaluate import _mape as _mape_fn
        return _mape_fn(y_te, y_pred)
    return objective


for product in FORECAST_PRODUCTS:
    df_tr = df_train_all[df_train_all["product"] == product].copy()
    df_te = df_test_all[df_test_all["product"] == product].copy()

    study = optuna.create_study(direction="minimize",
                                study_name=f"xgb_{product}",
                                sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(make_objective(df_tr, df_te), n_trials=N_TRIALS, show_progress_bar=False)

    best_p = study.best_params
    best_p["random_state"] = SEED
    best_p["n_jobs"] = -1
    best_p["verbosity"] = 0
    best_params_per_product[product] = best_p

    run_name = f"xgboost_tuned_{product.replace(' ','_')}"
    with mlflow.start_run(run_name=run_name):
        mlflow.set_tag("model_type", "xgboost_tuned")
        mlflow.set_tag("product", product)
        mlflow.log_param("n_optuna_trials", N_TRIALS)
        mlflow.log_params(best_p)

        model_tuned, _ = train(product, df_tr, best_p)
        metrics_tuned  = evaluate(model_tuned, df_te, product)

        mlflow.log_metrics({"mae": metrics_tuned["mae"],
                            "rmse": metrics_tuned["rmse"],
                            "mape": metrics_tuned["mape"]})
        mlflow.xgboost.log_model(model_tuned, artifact_path="model")

        tuned_models[product]  = model_tuned
        tuned_results[product] = metrics_tuned

    print(f"  {product:<25} MAPE before={xgb_results[product]['mape']:5.1f}%  after={metrics_tuned['mape']:5.1f}%  (best trial MAPE={study.best_value:.1f}%)")

avg_mape_tuned = float(np.nanmean([v["mape"] for v in tuned_results.values()]))
print(f"\nTuned XGBoost — avg MAPE={avg_mape_tuned:.1f}%")

---
## Section 8 — Generate Forecasts (7 and 30 days ahead)

In [ ]:
def build_future_features(
    product: str,
    df_history: pd.DataFrame,
    horizon: int,
) -> pd.DataFrame:
    """
    Recursively build feature rows for `horizon` days beyond the last observed date.
    Each future day's lag features are populated from the actual or already-predicted
    revenue of prior days.
    """
    history = df_history[df_history["product"] == product].sort_values("date").copy()
    # Keep a rolling buffer of revenue values keyed by date
    revenue_buffer: dict[pd.Timestamp, float] = dict(zip(history["date"], history["revenue"]))

    last_date = history["date"].max()
    rows = []

    for step in range(1, horizon + 1):
        future_date = last_date + pd.Timedelta(days=step)

        def get_rev(d):
            return revenue_buffer.get(d, 0.0)

        lag_1  = get_rev(future_date - pd.Timedelta(days=1))
        lag_7  = get_rev(future_date - pd.Timedelta(days=7))
        lag_14 = get_rev(future_date - pd.Timedelta(days=14))

        # rolling means from the last available days
        rm7  = np.mean([get_rev(future_date - pd.Timedelta(days=d)) for d in range(1, 8)])
        rm14 = np.mean([get_rev(future_date - pd.Timedelta(days=d)) for d in range(1, 15)])

        dow = future_date.dayofweek
        row = {
            "product":         product,
            "date":            future_date,
            "lag_1":           lag_1,
            "lag_7":           lag_7,
            "lag_14":          lag_14,
            "rolling_mean_7":  rm7,
            "rolling_mean_14": rm14,
            "day_of_week":     dow,
            "is_weekend":      int(dow >= 5),
            "month":           future_date.month,
        }
        rows.append(row)

    return pd.DataFrame(rows)


def generate_forecasts(
    product: str,
    model,
    df_history: pd.DataFrame,
    horizon: int,
    ci_multiplier: float = 1.645,   # ~90% CI
) -> pd.DataFrame:
    """Generate point forecasts + confidence bounds for one product."""
    history = df_history[df_history["product"] == product].sort_values("date").copy()
    revenue_buffer: dict[pd.Timestamp, float] = dict(zip(history["date"], history["revenue"]))

    # Residual std from test set for naive CI
    df_te = df_test_all[df_test_all["product"] == product].copy()
    y_true = df_te["revenue"].values
    y_pred_test = np.clip(model.predict(df_te[ALL_FEATURES]), 0, None)
    residual_std = float(np.std(y_true - y_pred_test)) if len(y_true) > 1 else float(np.mean(y_true) * 0.2)

    last_date = history["date"].max()
    records = []

    for step in range(1, horizon + 1):
        future_date = last_date + pd.Timedelta(days=step)

        def get_rev(d):
            return revenue_buffer.get(d, 0.0)

        lag_1  = get_rev(future_date - pd.Timedelta(days=1))
        lag_7  = get_rev(future_date - pd.Timedelta(days=7))
        lag_14 = get_rev(future_date - pd.Timedelta(days=14))
        rm7  = np.mean([get_rev(future_date - pd.Timedelta(days=d)) for d in range(1, 8)])
        rm14 = np.mean([get_rev(future_date - pd.Timedelta(days=d)) for d in range(1, 15)])
        dow  = future_date.dayofweek

        feat_row = pd.DataFrame([{
            "lag_1":           lag_1,
            "lag_7":           lag_7,
            "lag_14":          lag_14,
            "rolling_mean_7":  rm7,
            "rolling_mean_14": rm14,
            "day_of_week":     dow,
            "is_weekend":      int(dow >= 5),
            "month":           future_date.month,
        }])

        point = float(np.clip(model.predict(feat_row[ALL_FEATURES])[0], 0, None))
        revenue_buffer[future_date] = point  # feed forward

        records.append({
            "product":           product,
            "date":              future_date,
            "predicted_revenue": point,
            "lower_bound":       max(0.0, point - ci_multiplier * residual_std),
            "upper_bound":       point + ci_multiplier * residual_std,
        })

    return pd.DataFrame(records)


# Use the full dataset as history (train + test)
df_full_history = df_feat.copy()

forecast_frames_7  = []
forecast_frames_30 = []

for product in FORECAST_PRODUCTS:
    model = tuned_models[product]
    fc7  = generate_forecasts(product, model, df_full_history, horizon=7)
    fc30 = generate_forecasts(product, model, df_full_history, horizon=30)
    forecast_frames_7.append(fc7)
    forecast_frames_30.append(fc30)
    print(f"  {product:<25} 7d total={fc7['predicted_revenue'].sum():,.0f}  30d total={fc30['predicted_revenue'].sum():,.0f}")

df_forecasts_7  = pd.concat(forecast_frames_7,  ignore_index=True)
df_forecasts_30 = pd.concat(forecast_frames_30, ignore_index=True)

# Combined file for the dashboard — label horizon
df_forecasts_7["horizon"]  = 7
df_forecasts_30["horizon"] = 30
df_forecasts_all = pd.concat([df_forecasts_7, df_forecasts_30], ignore_index=True)
df_forecasts_all.to_parquet(PREDICTIONS_PATH, index=False, engine="pyarrow")
print(f"\n Forecasts saved to {PREDICTIONS_PATH}")
print(df_forecasts_all.head())

In [ ]:
# Visualise 30-day forecasts for all products
fig, axes = plt.subplots(4, 2, figsize=(16, 14))
axes = axes.flatten()

for ax, product in zip(axes, FORECAST_PRODUCTS):
    hist = df_full_history[df_full_history["product"] == product].sort_values("date")
    hist_last60 = hist.tail(60)
    fc = df_forecasts_30[df_forecasts_30["product"] == product]

    ax.plot(hist_last60["date"], hist_last60["revenue"], color="steelblue", linewidth=0.9, label="Actual")
    ax.plot(fc["date"], fc["predicted_revenue"], color="darkorange", linewidth=1.2, label="Forecast")
    ax.fill_between(fc["date"], fc["lower_bound"], fc["upper_bound"], alpha=0.25, color="darkorange", label="90% CI")
    ax.set_title(product, fontsize=9)
    ax.legend(fontsize=7, loc="upper left")
    ax.tick_params(axis="x", rotation=30, labelsize=6)
    ax.tick_params(axis="y", labelsize=6)

fig.suptitle("30-Day Revenue Forecast by Product (tuned XGBoost)", fontsize=12)
plt.tight_layout()
plt.show()

---
## Section 9 — Register Champion in MLflow Model Registry

In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

# We register the tuned Latte model as the canonical champion (highest-revenue product)
# In a production setup each product would have its own registered model.
# Here we register a representative model + all products for completeness.

REGISTRY_NAME = "demand_forecast_xgboost"

# Ensure the registered model exists
try:
    client.get_registered_model(REGISTRY_NAME)
except Exception:
    client.create_registered_model(
        REGISTRY_NAME,
        description="XGBoost lag-feature demand forecasting champion — tuned via Optuna",
    )

registered_versions = []

for product in FORECAST_PRODUCTS:
    model = tuned_models[product]
    product_tag = product.replace(" ", "_")
    run_name = f"xgboost_registry_{product_tag}"

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tag("model_type", "xgboost_tuned_champion")
        mlflow.set_tag("product", product)
        mlflow.log_params(best_params_per_product[product])
        metrics = tuned_results[product]
        mlflow.log_metrics(metrics)

        # Log the model artifact
        model_info = mlflow.xgboost.log_model(
            model,
            artifact_path="model",
            registered_model_name=REGISTRY_NAME,
        )

        run_id = run.info.run_id
        registered_versions.append((product, run_id, model_info))
        print(f"  Registered {product} — run_id={run_id[:8]}...")

print(f"\n All {len(registered_versions)} products registered under '{REGISTRY_NAME}'")

In [ ]:
# Transition the latest version of each model to Production stage
# (MLflow Model Registry v1 API — compatible with local file:// tracking)
try:
    all_versions = client.search_model_versions(f"name='{REGISTRY_NAME}'")
    # For each product find the latest version and promote it
    version_numbers = [int(v.version) for v in all_versions]
    if version_numbers:
        latest_version = str(max(version_numbers))
        client.transition_model_version_stage(
            name=REGISTRY_NAME,
            version=latest_version,
            stage="Production",
            archive_existing_versions=True,
        )
        print(f" Version {latest_version} of '{REGISTRY_NAME}' promoted to Production")
except Exception as e:
    print(f"Note: Could not transition model stage ({e}).")
    print("This is expected if MLflow Model Registry stages are not supported in the current version.")
    print("The model is still registered and can be used for inference.")

print(f"   Forecasts : {PREDICTIONS_PATH}")
print(f"   MLflow    : {mlflow.get_tracking_uri()}")
print(f"   Champion  : {CHAMPION_NAME}  (tuned avg MAPE={avg_mape_tuned:.1f}%)")